In [ ]:
# Carico la cartella one drive in cui ci sono:
# cartella video nutrition estimation / video annotation nutrition estimation
# cartella video nutrition change / video annotation nutrition change
# cartella video image nutrition estimation/ video annotation image estimation

# come ho scaricato i video?
# partendo dai file json ho lanciato lo script downloader sugli id estratti
# dei video scaricati ho fatto un samplig a N=128 FRAME
# caricato su one drive ed adesso linko da questo notebook
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Verifico scheda video e versione compilatore NVCC disponibile nell'ambiente**
- in questo caso A100 con 40Gb di VRAM

In [2]:
!nvidia-smi
!nvcc --version

Mon Jul 13 20:24:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             46W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
import os

# scarico la libreria trasformers utilizzata nella Repo LLaVA-NeXT
#(4.37.2 è l'ultima compatibile con apply_chunking_to_forward e LlavaLlamaForCausalLM)
!pip install "transformers==4.37.2" --quiet

# clono il repo ufficiale lmms-lab/LLaVA-NeXT
if not os.path.exists("/content/LLaVA-NeXT"):
    !git clone https://github.com/LLaVA-VL/LLaVA-NeXT.git /content/LLaVA-NeXT

%cd /content/LLaVA-NeXT

# installo dipendenze del repo (senza sovrascrivere la versione di transformers pinnata)
!pip install -e ".[train]" --no-deps --quiet
!pip install av accelerate peft timm einops sentencepiece --quiet

print("Setup completato — transformers:", end=" ")
import transformers; print(transformers.__version__)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 190.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 163.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.3.0 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.37.2 which is incompatible.
Cloning into '/content/LLaVA-NeXT'...
remote: Enumerating objects: 6643, done.
remote: Counting objects: 100% (34/34), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 6643 (delta 22), reused 12 (delta 12), pack-reused 6609 (from 3)
Receiving objects: 100% (6643/6643), 121.23 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (4129/4129), done.
/content/LLaVA-N

In [ ]:
import sys
import torch

# aggiungo il repo su Google Drive
if "/content/LLaVA-NeXT" not in sys.path:
    sys.path.insert(0, "/content/LLaVA-NeXT")

# Loader ufficiale lmms-lab — carica pesi, tokenizer e image_processor
# con le chiavi corrette per LLaVA-Video-7B-Qwen2
from llava.model.builder import load_pretrained_model

model_path = "lmms-lab/LLaVA-Video-7B-Qwen2"
model_name = "llava_qwen"   # nome architettura richiesto dal builder


# carico il modello
tokenizer, model, image_processor, max_length = load_pretrained_model(
    model_path,
    None,           # model_base: None per modelli stand-alone
    model_name,
    torch_dtype="bfloat16",
    device_map="auto",
    attn_implementation="sdpa",
)


# mettiamo in modalità evaluation
model.eval()
device = next(model.parameters()).device
print(f"LLaVA-Video-7B-Qwen2 caricato su {device}")


/content/LLaVA-NeXT/llava/model/llava_arch.py:215: SyntaxWarning: "is not" with 'int' literal. Did you mean "!="?
  if slower_img_feat is not 0:


Please install pyav to use video processing functions.
OpenCLIP not installed
Loaded LLaVA model: lmms-lab/LLaVA-Video-7B-Qwen2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings.patch_embedding.bias: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  module._load_from_state_dict(*args)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:621: UserWarning: for vision_model.embeddings

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/248 [00:00<?, ?B/s]

Model Class: LlavaQwenForCausalLM
LLaVA-Video-7B-Qwen2 caricato su cuda:0


In [ ]:
import av
import numpy as np


# funzione per capionare i frame video
# A differenza di Video-LLaMA2 che usava la sua funzione interna mm_infer, con Hugging Face useremo decord per campionare in modo uniforme i frame del video da passare al processore di LLaVA.
def sample_video_frames(video_path, max_frames=64):
    """
    Legge il video con PyAV ed estrae uniformemente max_frames frame.
    LLaVA-Video usa PyAV (non decord) nel codebase ufficiale.
    Restituisce array numpy (num_frames, H, W, 3) in formato RGB.
    """
    container = av.open(video_path)
    stream = container.streams.video[0]
    total_frames = stream.frames

    # Se il numero di frame non è nel metadato, lo contiamo
    if total_frames == 0:
        total_frames = sum(1 for _ in container.decode(video=0))
        container.seek(0)

    indices = set(np.linspace(0, total_frames - 1, max_frames, dtype=int).tolist())

    frames = []
    for i, frame in enumerate(container.decode(video=0)):
        if i in indices:
            frames.append(frame.to_ndarray(format="rgb24"))
        if len(frames) == max_frames:
            break

    container.close()
    return np.stack(frames)  # (num_frames, H, W, 3)


In [ ]:
import re

# librerie per la gestione dei prompt ben formati da passare al modello
from llava.conversation import conv_templates
from llava.constants import DEFAULT_IMAGE_TOKEN, IMAGE_TOKEN_INDEX
from llava.mm_utils import tokenizer_image_token



# funzione per la costruzione del prompt ben formato
# LLaVA richiede un token speciale (o un flag del processore) per capire dove inserire il video all'interno della conversazione. Il formato standard della chat per la versione Hugging Face usa la struttura <video>\nUSER: ... \nASSISTANT:.
def build_llava_video_prompt(question, choices, tokenizer):
    """
    Costruisce il prompt per LLaVA-Video-7B-Qwen2 seguendo il template
    ufficiale 'qwen_1_5' del repo lmms-lab/LLaVA-NeXT.
    Il token <image> viene usato anche per i video nel codebase ufficiale.
    """
    choices_text = "\n".join([f"{i}. {c}" for i, c in enumerate(choices)])

    user_message = (
        f"{DEFAULT_IMAGE_TOKEN}\n"  #spesso <image> amche se video, fa capire al modello contenuto che deve processare in input
        f"Question: {question}\n\n"
        f"Choices:\n{choices_text}\n\n"
        f"Answer with ONLY the digit (0-{len(choices)-1}) of the correct choice."
    )

    # copio il template del modello perchè senno il modello non risponde bene 
    conv = conv_templates["qwen_1_5"].copy()        
    conv.append_message(conv.roles[0], user_message)
    conv.append_message(conv.roles[1], None)
    prompt = conv.get_prompt()

    # Converte il testo formattato in una sequenza di numeri interi, anche il token di default dell'immagine
    input_ids = tokenizer_image_token(
        prompt, tokenizer, IMAGE_TOKEN_INDEX, return_tensors="pt"
    ).unsqueeze(0)

    return input_ids

# per parserizzare le risposte ritornate dal modello
def parse_answer(response_text, num_choices):
    matches = re.findall(r'\b([0-9])\b', response_text)
    for m in reversed(matches):
        idx = int(m)
        if 0 <= idx < num_choices:
            return idx
    return -1


In [ ]:
import os


# Inferenza con LLaVA-Video-7B-Qwen2 usando il pipeline ufficiale lmms-lab.
def inference_on_videos_llava(model, tokenizer, image_processor, annotations, video_dir):
   
    results = []


    # per ogni annotation dei task prendo video id, domande, scelte possibili e scelta corretta
    for key, item in annotations.items():
        video_id   = item["inputs"]["video 1"]["id"]
        question   = item["question"]
        choices    = item["choices"]
        correct    = item["correct_idx"]
        video_path = os.path.join(video_dir, f"{video_id}_sampled.mp4")

        print(f"▶ {key} | video: {video_id}")

        try:
            if not os.path.exists(video_path):
                raise FileNotFoundError(f"Video non trovato: {video_path}")

            # estraggo 64 frame in RGB (potrei aumentare a 128)
            frames = sample_video_frames(video_path, max_frames=64)

            # processo i frame con l'image_processor ufficiale
            video_tensor = image_processor.preprocess(frames, return_tensors="pt")["pixel_values"]
            video_tensor = video_tensor.to(device=device, dtype=torch.bfloat16)

            # costruisco input_ids con il template Qwen1.5
            input_ids = build_llava_video_prompt(question, choices, tokenizer)
            input_ids = input_ids.to(device)

            # genero la risposta passando l'input visivo e l'input testuale (entrambi pre processati)
            with torch.no_grad():
                output_ids = model.generate(
                    input_ids,
                    images=[video_tensor],
                    image_sizes=None,
                    modalities=["video"],
                    do_sample=False,
                    temperature=None,
                    max_new_tokens=32,
                    use_cache=True,
                )

            # decodifico l'output COMPLETO (con special tokens) e poi estrai
            full_output = tokenizer.decode(output_ids[0], skip_special_tokens=False)

            # il template Qwen1.5 usa "<|im_start|>assistant\n" come separatore (prendo la risposta pulita rimuovendo i caratteri spuri)
            assistant_marker = "<|im_start|>assistant\n"
            if assistant_marker in full_output:
                response = full_output.split(assistant_marker)[-1]
                response = response.replace("<|im_end|>", "").strip()
            else:
                # Fallback: decodifica senza special tokens e prendi l'ultimo token non-prompt
                response = tokenizer.decode(output_ids[0], skip_special_tokens=True).strip()

            predicted  = parse_answer(response, len(choices))
            is_correct = (predicted == correct)

            print(f"   Risposta grezza: {repr(response)}")
            print(f"   Predetto: {predicted} | Corretto: {correct} | {'✅' if is_correct else '❌'}\n")

        except Exception as e:
            response   = f"ERRORE: {e}"
            predicted  = -1
            is_correct = False
            print(f"   ⚠️ Errore: {e}\n")

        q_type = question.split("highest")[-1].strip().rstrip("?").strip() if "highest" in question else "unknown"

        # accumulo i risultati
        results.append({
            "key":           key,
            "video_id":      video_id,
            "question":      question,
            "q_type":        q_type,
            "correct_idx":   correct,
            "predicted_idx": predicted,
            "is_correct":    is_correct,
            "raw_response":  response,
        })

    return results


In [8]:

import json
import os

# cartella del progetto
BASE_DRIVE_PATH = "/content/drive/MyDrive/Progetto Deep Learning 1"

# Definisco un task come:
# coppia video annotazioni con domande
# in particolare nel dataset abbiamo
# domande sul valore nutrizionale totale su un video / cartella di video 
# domande su come cambia il valore nutrizionale di un video  post inserimento di un ingrediente /cartella di video 
# domande su ingredienti inseriti in un determinato frame di un video  /cartella di video

tasks = {
    "nutrition_estimation": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_video_nutrition_estimation.json"),
        "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Nutrition Estimation Sampled"),
    },
    "nutrition_change": {
        "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_nutrition_change.json"),
        "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Nutrition Change Sampled"),
    },
    # "image_nutrition_estimation": {
    #     "annotations": os.path.join(BASE_DRIVE_PATH, "nutrition_image_nutrition_estimation.json"),
    #     "video_dir":   os.path.join(BASE_DRIVE_PATH, "Videos Image Nutrition Estimation Sampled"),
    # },
}


In [9]:
import json
import pandas as pd

# =====================================================================
# LOOP SU TUTTI I TASK: carica annotation, esegue inferenza, accumula
# =====================================================================
all_results = []

for task_name, paths in tasks.items():
    print(f"\n{'='*60}")
    print(f"  TASK: {task_name}")
    print(f"{'='*60}")

    with open(paths["annotations"], "r") as f:
        annotations = json.load(f)
    print(f"  Caricate {len(annotations)} annotazioni.")

    task_results = inference_on_videos_llava(
        model=model,
        tokenizer=tokenizer,
        image_processor=image_processor,
        annotations=annotations,
        video_dir=paths["video_dir"]
    )

    for r in task_results:
        r["task"] = task_name

    all_results.extend(task_results)
    print(f"  Task '{task_name}' completato: {len(task_results)} campioni")

df_all = pd.DataFrame(all_results)
print(f"\nInferenza completata su tutti i task — totale campioni: {len(df_all)}")



  TASK: nutrition_estimation
  Caricate 50 annotazioni.
▶ nutrition_video_nutrition_estimation_0 | video: P03-20240217-131219


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:392: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `None` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:397: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:407: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


   Risposta grezza: '0.'
   Predetto: 0 | Corretto: 3 | ❌

▶ nutrition_video_nutrition_estimation_1 | video: P03-20240218-190556
   Risposta grezza: '3'
   Predetto: 3 | Corretto: 0 | ❌

▶ nutrition_video_nutrition_estimation_2 | video: P03-20240217-131219
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_3 | video: P08-20240614-184945
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_4 | video: P03-20240217-131219
   Risposta grezza: '1.'
   Predetto: 1 | Corretto: 1 | ✅

▶ nutrition_video_nutrition_estimation_5 | video: P08-20240622-141902
   Risposta grezza: '4'
   Predetto: 4 | Corretto: 4 | ✅

▶ nutrition_video_nutrition_estimation_6 | video: P08-20240617-130401
   Risposta grezza: '2'
   Predetto: 2 | Corretto: 2 | ✅

▶ nutrition_video_nutrition_estimation_7 | video: P03-20240217-131219
   Risposta grezza: '3'
   Predetto: 3 | Corretto: 3 | ✅

▶ nutrition_video_nutrition_estimation_8 | v

In [10]:

# ── Analisi delle performance globali ─────────────────────────────────────────

df = df_all.copy()

total    = len(df)
correct  = df["is_correct"].sum()
accuracy = correct / total * 100
random_baseline = 100 / 5  # 5 scelte sempre

print("=" * 60)
print(f"  ACCURACY GLOBALE  : {accuracy:.1f}%  ({correct}/{total})")
print(f"  Baseline random   : {random_baseline:.1f}%")
print("=" * 60)

# Accuracy per task
print("\nAccuracy per task:")
acc_by_task = (
    df.groupby("task")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_task["accuracy_%"] = (acc_by_task["corrette"] / acc_by_task["totali"] * 100).round(1)
print(acc_by_task.to_string())

# Accuracy per tipo di domanda
print("\nAccuracy per tipo di domanda:")
acc_by_type = (
    df.groupby("q_type")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_type["accuracy_%"] = (acc_by_type["corrette"] / acc_by_type["totali"] * 100).round(1)
print(acc_by_type.to_string())

# Accuracy per video
print("\nAccuracy per video:")
acc_by_video = (
    df.groupby("video_id")["is_correct"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "corrette", "count": "totali"})
)
acc_by_video["accuracy_%"] = (acc_by_video["corrette"] / acc_by_video["totali"] * 100).round(1)
print(acc_by_video.to_string())

# Campioni con predicted_idx == -1 (risposta non parsata)
n_invalid = (df["predicted_idx"] == -1).sum()
print(f"\nRisposte non parsate (predicted_idx = -1): {n_invalid}/{total}")

# Salva i risultati su CSV nel Drive
output_csv = "/content/drive/MyDrive/inference_results_llava_all_tasks.csv"
df.to_csv(output_csv, index=False)
print(f"\nRisultati salvati in: {output_csv}")


  ACCURACY GLOBALE  : 44.0%  (44/100)
  Baseline random   : 20.0%

Accuracy per task:
                      corrette  totali  accuracy_%
task                                              
nutrition_change            15      50        30.0
nutrition_estimation        29      50        58.0

Accuracy per tipo di domanda:
                         corrette  totali  accuracy_%
q_type                                               
calories in this recipe         8      14        57.1
carbs in this recipe            9      13        69.2
fat in this recipe              5      10        50.0
protein in this recipe          7      13        53.8
unknown                        15      50        30.0

Accuracy per video:
                     corrette  totali  accuracy_%
video_id                                         
P01-20240203-130505         0       2         0.0
P01-20240203-135502         1       3        33.3
P01-20240204-121042         0       2         0.0
P03-20240217-131219         3 